In [1]:
import os

In [2]:
%pwd

'd:\\Work\\Internship\\Inuron\\mushroom_classification\\test'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Work\\Internship\\Inuron\\mushroom_classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [6]:
@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    model_name: str
    n_estimators: int
    TARGET_COLUMN: str

In [7]:
from src.constants import *
from src.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH,
            schema_filepath=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.RandomForestClassifier
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            model_name=config.model_name,
            n_estimators=params.n_estimators,
            TARGET_COLUMN=schema.name
        )

        return model_trainer_config

In [9]:
import pandas as pd
import os
from src import logger
from sklearn.ensemble import RandomForestClassifier
import joblib

In [10]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)

        train_x = train_data.drop([self.config.TARGET_COLUMN], axis=1)
        train_y = train_data[[self.config.TARGET_COLUMN]]


        clf = RandomForestClassifier(n_estimators=self.config.n_estimators)
        clf.fit(train_x, train_y)

        joblib.dump(clf, os.path.join(self.config.root_dir, self.config.model_name))

In [11]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2024-04-25 13:33:41,997: INFO: common: yaml file: config\config.yaml loaded successfully]
[2024-04-25 13:33:41,999: INFO: common: yaml file: params.yaml loaded successfully]
[2024-04-25 13:33:42,003: INFO: common: yaml file: schema.yaml loaded successfully]
[2024-04-25 13:33:42,005: INFO: common: created directory at: artifacts]
[2024-04-25 13:33:42,006: INFO: common: created directory at: artifacts/model_trainer]


d:\Work\Internship\Inuron\mushroom_classification\mushroom_env\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
